In [4]:
import pandas as pd
import re
import unicodedata

# ==============================
# 1. LOAD DATA
# ==============================

bushtel = pd.read_csv(
    "bushtel_communities_2024.csv",
    encoding="utf-8-sig"
)

abs_population = pd.read_csv(
    "2021Census_I01A_NT_ILOC.csv",
    encoding="utf-8-sig"
)

abs_geography = pd.read_excel(
    "2021Census_geog_desc_1st_2nd_3rd_release.xlsx",
    sheet_name="2021_ASGS_Indigenous_Structures",
    dtype=str,
    engine="openpyxl"
)

# ==============================
# 2. PREPARE ABS POPULATION
# ==============================

abs_population = abs_population[
    [
        "ILOC_CODE_2021",
        "Tot_p_TotP",
        "Tot_p_Indig_P"
    ]
].rename(columns={
    "Tot_p_TotP": "abs_total_population_2021",
    "Tot_p_Indig_P": "abs_indigenous_population_2021"
})

# ==============================
# 3. PREPARE ABS GEOGRAPHY
# ==============================

abs_geography = abs_geography[
    abs_geography["ASGS_Structure"] == "ILOC"
][
    [
        "Census_Code_2021",
        "Census_Name_2021"
    ]
].rename(columns={
    "Census_Code_2021": "ILOC_CODE_2021",
    "Census_Name_2021": "iloc_name"
})

# ==============================
# 4. MERGE ABS DATA
# ==============================

abs_iloc = abs_population.merge(
    abs_geography,
    on="ILOC_CODE_2021",
    how="left"
)

# ==============================
# 5. NORMALISE NAMES
# ==============================

def normalise_name(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().upper()

    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")

    value = re.sub(r"[-–—]", " ", value)
    value = re.sub(r"[^A-Z0-9 ]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value


bushtel["match_key"] = bushtel["community_name"].apply(
    normalise_name
)

abs_iloc["match_key"] = abs_iloc["iloc_name"].apply(
    normalise_name
)

# ==============================
# 6. MERGE BUSHTEL WITH ABS
# ==============================

result = bushtel.merge(
    abs_iloc[
        [
            "match_key",
            "ILOC_CODE_2021",
            "iloc_name",
            "abs_total_population_2021",
            "abs_indigenous_population_2021"
        ]
    ],
    on="match_key",
    how="left"
)

# ==============================
# 7. CREATE POPULATION_FINAL
# ==============================

# Giữ nguyên population_count là population gốc
result["population_final"] = result["population_count"]

missing_population = result["population_count"].isna()
abs_match = result["abs_total_population_2021"].notna()

accepted_match = missing_population & abs_match

# Bổ sung ABS chỉ khi population gốc bị thiếu
result.loc[
    accepted_match,
    "population_final"
] = result.loc[
    accepted_match,
    "abs_total_population_2021"
]

# ==============================
# 8. RECORD SOURCE
# ==============================

result["population_final_source"] = "Unavailable"
result["population_match_status"] = "No population found"

# Dân số gốc từ BushTel
has_bushtel_population = result["population_count"].notna()

if "population_source" in result.columns:
    result.loc[
        has_bushtel_population,
        "population_final_source"
    ] = result.loc[
        has_bushtel_population,
        "population_source"
    ].fillna("BushTel")
else:
    result.loc[
        has_bushtel_population,
        "population_final_source"
    ] = "BushTel"

result.loc[
    has_bushtel_population,
    "population_match_status"
] = "BushTel population"

# Dân số bổ sung từ ABS
result.loc[
    accepted_match,
    "population_final_source"
] = "ABS Census 2021 ILOC"

result.loc[
    accepted_match,
    "population_match_status"
] = "ABS exact-name match"

# ==============================
# 9. REMOVE TEMPORARY COLUMN
# ==============================

result = result.drop(
    columns=["match_key"],
    errors="ignore"
)

# ==============================
# 10. CREATE CLEAN DATASET
# ==============================

required_columns = [
    "community_id",
    "community_name",
    "community_type",
    "region",
    "latitude",
    "longitude",
    "population_count",
    "population_final",
    "population_final_source",
    "population_match_status",
    "ILOC_CODE_2021",
    "iloc_name"
]

# Chỉ giữ các cột thực sự tồn tại
clean_columns = [
    column for column in required_columns
    if column in result.columns
]

clean_result = result[clean_columns].copy()

# Đổi tên population_count thành population
clean_result = clean_result.rename(columns={
    "population_count": "population",
    "population_final_source": "population_source"
})

# ==============================
# 11. SAVE CLEAN FILE
# ==============================

clean_result.to_csv(
    "bushtel_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==============================
# 12. DISPLAY RESULTS
# ==============================

print("Completed successfully.")
print("Output file: bushtel_clean.csv")
print("Total records:", len(clean_result))

print("\nColumns kept:")
print(clean_result.columns.tolist())

print(
    "\nRecords with original population:",
    clean_result["population"].notna().sum()
)

print(
    "Records with final population:",
    clean_result["population_final"].notna().sum()
)

print(
    "Population added from ABS:",
    accepted_match.sum()
)

print(
    "Population still missing:",
    clean_result["population_final"].isna().sum()
)

Completed successfully.
Output file: bushtel_clean.csv
Total records: 792

Columns kept:
['community_id', 'community_name', 'community_type', 'latitude', 'longitude', 'population', 'population_final', 'population_source', 'population_match_status', 'ILOC_CODE_2021', 'iloc_name']

Records with original population: 457
Records with final population: 468
Population added from ABS: 11
Population still missing: 324
